In [1]:
import fine as fn
import pyomo.environ as pyomo
import pandas as pd 

# Step 1: Define the Energy System Model
esM = fn.EnergySystemModel(
    locations={"A"},
    onlycommodities={"electricity", "hydrogen"},
    onlycommodityUnitsDict={"electricity": "GW", "hydrogen": "kg"},
    onlymaterials={"steel", "copper", "iron"},
    onlymaterialUnitsDict={"steel": "tons", "copper": "kg", "iron": "kg"}
)
#esM.pyM = pyomo.ConcreteModel()

In [2]:
# energyCommoditySet = {'electricty'}
# materialCommoditySet = {'steel'}
# commodityUnitDict = {'electricty': 'kWh', 'steel': 't'}
# print(processedMaterialIntensity[('location1', 2020, 'material1')]) 

# print(processedMaterialIntensity["location1"]["2020"]["material1"]) 

In [3]:
# Check if commodity declarations work
print("only Materials Units Dict:", esM.onlymaterialUnitsDict)
print("only Commodity Units Dict:", esM.onlycommodityUnitsDict)
print("Commodities:", esM.commodities)
print("Commodity Units Dict:", esM.commodityUnitsDict)

only Materials Units Dict: {'steel': 'tons', 'copper': 'kg', 'iron': 'kg'}
only Commodity Units Dict: {'electricity': 'GW', 'hydrogen': 'kg'}
Commodities: ['electricity', 'hydrogen', 'copper', 'iron', 'steel']
Commodity Units Dict: {'electricity': 'GW', 'hydrogen': 'kg', 'copper': 'kg', 'iron': 'kg', 'steel': 'tons'}


In [4]:
# Step 2: Add a Energy Source Component that Requires Materials                             
esM.add(
    fn.Source(
        esM=esM, 
        name="Wind Turbines",
        commodity="electricity",
        hasCapacityVariable=True,
        materialIntensity={
            'A': {0: {'iron':1.0, 'copper': 5.2, 'steel': 3.1}, 1: {'iron':1.0, 'copper': 5.3, 'steel': 3.2}},
            'B': {0: {'iron':1.0, 'copper': 4.8, 'steel': 2.9}, 1: {'iron':1.0, 'copper': 4.9, 'steel': 3.0}}
            },  # Materials required for commissioning
        # materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
)

# # Add a Energy Storage Component that Requires Materials
# esM.add(
#     fn.Storage(
#         esM=esM,
#         name="Battery",
#         commodity="electricity",
#         chargeEfficiency=0.9,
#         dischargeEfficiency=0.9,
#         materialIntensity={
#             'A': {0: {'copper': 5.2, 'steel': 3.1}, 1: {'copper': 5.3, 'steel': 3.2}},
#             'B': {0: {'copper': 4.8, 'steel': 2.9}, 1: {'copper': 4.9, 'steel': 3.0}}
#             },  # Materials required for commissioning
#         # materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
#     )
# ) 


In [5]:
# Step 3: Add Material Source 
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Supply",
        commodity="steel",
        hasCapacityVariable=True,
        material=True,
    )
)

source = esM.add(
    fn.Source(
        esM=esM, 
        name="Copper Supply",
        commodity="copper",
        hasCapacityVariable=True,
        material=True,
    )
)
#esM.source.commodity

In [6]:
# Step 4: Add Energy Sink Component that consumes Energy 
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=50,
        
    )
)

In [7]:
# Step 5: Add Material Sink that consumes Materials 
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Steel demand",
        hasCapacityVariable=False,
        commodity="steel",
        material=True,      # automized recognition of materials
    )
)

In [8]:
esM.aggregateTemporally(numberOfTypicalPeriods=30)


Clustering time series data with 30 typical periods and 24 time steps per period 
further clustered to 12 segments per period...
		(1.4513 sec)



In [9]:
esM.declareOptimizationProblem()

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
{'steel', 'iron', 'copper'}
	declaring variables... 
	declaring constraints... 
ERROR: Rule failed when generating expression for Constraint
ConstrOperation5_srcSnk with index ('A', 'Wind Turbines', 'steel', 0):
KeyError: "Index '('A', 'Wind Turbines', 0, 1, 2)' is not valid for indexed
component 'op_srcSnk'"
ERROR: Constructing component 'ConstrOperation5_srcSnk' from data=None failed:
        KeyError: "Index '('A', 'Wind Turbines', 0, 1, 2)' is not valid for
        indexed component 'op_srcSnk'"


KeyError: "Index '('A', 'Wind Turbines', 0, 1, 2)' is not valid for indexed component 'op_srcSnk'"

In [ ]:
esM.optimize()

------------------------------------------------------------------------------------------------------------------------
## Test material loop

In [ ]:
esM.componentModelingDict.values()

In [ ]:
def checkAndSetMaterialIntensity(esM, materialIntensity, locations, investmentPeriods, onlymaterials):
    # Initialisiere ein leeres Dictionary für die verarbeiteten Materialintensitäten
    processedMaterialIntensity = {}
 
    # Iteriere über alle Kombinationen von Locations, Investitionsperioden und Materialien
    for location in locations:
        for ip in investmentPeriods:
            for onlymaterial in onlymaterials:
                # Berechne das Investitionsjahr (basierend auf Startjahr und Periode)
                _ip = int(esM.startYear + ip * esM.investmentPeriodInterval)
               
                # Überprüfe, ob die Materialintensität für diese Kombination vorhanden ist
                if location in materialIntensity and _ip in materialIntensity[location] and onlymaterial in materialIntensity[location][_ip]:
                    # Setze die Materialintensität für diese Kombination
                    processedMaterialIntensity[(location, _ip, onlymaterial)] = materialIntensity[location][_ip][onlymaterial]                  
                else:
                    # Fehlerbehandlung: Fehlende Daten oder ungültige Kombination
                    raise ValueError(f"Missing material intensity for Location: {location}, IP: {_ip}, Material: {onlymaterial}")
   
    return processedMaterialIntensity

# Beispiel-Daten
esM = type("esM", (object,), {"startYear": 2020, "investmentPeriodInterval": 5})  # Beispiel für das esM-Objekt
materialIntensity = {
    'location1': {
        2020: {'material1': 5.2, 'material2': 3.1},
        2025: {'material1': 5.3, 'material2': 3.2},
    },
    'location2': {
        2020: {'material1': 4.8, 'material2': 2.9},
        2025: {'material1': 4.9, 'material2': 3.0},
    }
}
locations = ['location1', 'location2']
investmentPeriods = [0, 1]  # 0 für 2020 und 1 für 2025
onlymaterials = ['material1', 'material2']
 
# Funktion aufrufen
processedMaterialIntensity = checkAndSetMaterialIntensity(esM, materialIntensity, locations, investmentPeriods, onlymaterials)
 
print(processedMaterialIntensity[('location1', 2020, 'material1')])
 
#print(processedMaterialIntensity["location1"]["2020"]["material1"])


In [ ]:
processedMaterialIntensity 

In [ ]:
materialIntensity["location1"][2020]["material2"]

# lhs = sum(
#     opVar[loc, compName, ip, p, t]
#     for p in esM.typicalPeriods[ip]
#     for t in esM.hoursPerSegment[ip]
# )
processedMaterialIntensity[('location1', 2020, 'material1')]

rhs = 1.0 * processedMaterialIntensity[ip][loc][mat]

In [ ]:
d= {
    0: pd.Series(
        data=[0.0],
        index=["EnergyLand"],
        dtype="float64"
    )
}

print(d[0]["EnergyLand"])